# Subnetwork Extraction and Model Mask Generation Demo

This notebook demonstrates how to:
1. Extract a clean, duplicate-free list of upstream reach IDs starting from any arbitrary segment (for use as a mask in `t-route` execution) using the new `get_upstream_mask()` utility.
2. Visualize topological structure (DAG) of selected subgraphs (specifically USGS gages like Cahaba River, Walnut Creek, and Mulberry Creek) using the recursive ASCII tree printer.

These tasks support diagnostic subsetting of stream channel networks from NWM domain data.

### 1. Setup Environment and Load Data
First, we set up repository paths and import the `troute_network` package modules.

In [ ]:
import os
import sys

# Set high recursion limit for deep networks
sys.setrecursionlimit(6000)

# Resolve path to the test data directory (one level up from notebooks/)
root = os.path.dirname(os.path.abspath(''))
geo_input_folder = os.path.join(root, 'test_data')

import troute_network.nhd_network_utilities as nnu
import troute_network.recursive_print as rp
from troute_network import get_upstream_mask

print('Environment ready.')

In [ ]:
# Determine if the large CONUS RouteLink NetCDF file is present on the system
conus_nc_path = os.path.join(geo_input_folder, 'Channels', 'RouteLink_CONUS.nwm.v3.0.20.nc')

if os.path.exists(conus_nc_path):
    print('Found CONUS RouteLink file. Loading CONUS Full Resolution network...')
    active_network = 'CONUS_FULL_RES_v20'
else:
    print('CONUS RouteLink file not found. Falling back to Brazos & Lower Colorado subset...')
    active_network = 'Brazos_LowerColorado_ge5'

data, values = nnu.set_networks(
    supernetwork=active_network,
    geo_input_folder=geo_input_folder,
    verbose=False,
    debuglevel=-1
)

connections = values[0]
terminal_keys = values[4]
circular_keys = values[6]
terminal_keys_super = terminal_keys - circular_keys
terminal_code = data['terminal_code']

print(f'\nLoaded {active_network} with {len(connections)} total segments.')
print(f'Found {len(terminal_keys_super)} independent subnetworks.')

### 2. Extract a Clean Upstream Reach Mask for a Target Gage / Outlet
We will inspect our target gages in Alabama if the CONUS network is loaded (Walnut Creek, Mulberry Creek, and Cahaba River). We select one target to extract its upstream drainage mask and write it to a file.

In [ ]:
# Define target mapping for the three AL gages of interest
al_gages = {
    'Cahaba River at Centreville, AL (USGS 02424000)': 21661814,
    'Walnut Creek above Clanton, AL (USGS 02408150)': 22274808,
    'Mulberry Creek at Jones, AL (USGS 02422500)': 21676818
}

if active_network == 'CONUS_FULL_RES_v20':
    print('Inspecting target AL gages:')
    for name, link_id in al_gages.items():
        print(f' - {name} -> NWM LinkID: {link_id}')
    
    # Select Walnut Creek above Clanton as default; uncomment other lines for those locations
    target_name = 'Walnut Creek above Clanton, AL (USGS 02408150)'
    # target_name = 'Mulberry Creek at Jones, AL (USGS 02422500)'
    # target_name = 'Cahaba River at Centreville, AL (USGS 02424000)'
    
    target_outlet = al_gages[target_name]
    print(f'\nSelected Target: {target_name} (LinkID: {target_outlet})')
else:
    # Fallback to Brazos dataset terminal keys
    target_outlet = sorted(list(terminal_keys_super))[0]
    target_name = f'Brazos Subnetwork Terminal Outlet (LinkID: {target_outlet})'
    print(f'Using default fallback target: {target_name}')

# Extract upstream mask
upstream_mask = get_upstream_mask(target_outlet, connections, terminal_code)

print(f'\nTotal reaches in subnetwork: {len(upstream_mask)}')
print(f'First 15 reach IDs in subnetwork: {sorted(list(upstream_mask))[:15]}')

In [ ]:
# Save the clean reach list to a text file (one ID per line)
mask_filename = os.path.join(root, 'test_data', f'mask_{target_outlet}.txt')
with open(mask_filename, 'w') as f:
    for reach_id in sorted(upstream_mask):
        f.write(f'{reach_id}\n')

print(f'Saved clean mask file to: {mask_filename}')
print('This file is now ready for use in model configurations.')

### 3. Generate ASCII Tree Representations for the Target Subgraph
We print the topological ASCII representation starting downstream at the target reach.

In [ ]:
# Print the ASCII tree structure for our selected subnetwork
print(f'Topological tree structure starting from outlet {target_name} (LinkID: {target_outlet}):')
rp.print_connections(
    terminal_keys={target_outlet},
    up_connections=connections,
    down_connections=connections,
    terminal_code=terminal_code
)